# Model 2 — Frozen ModernBERT embeddings + classical heads

Predicts the **6-vector of compression perplexities** from prompt text.

- Data pulled live from GitHub (`perplexity_wide_complete.csv`).
- Targets in **log space**; inverted to raw perplexity for reporting.
- **Stratified random 70/30 split by dataset**, `random_state=42`.
- Trained model saved to `artifacts/`.

Requires `kv_common.py` in the same folder.

## Colab setup

Run this cell **first**. It clones the repo (so `kv_common.py` is available),
installs packages, and optionally mounts Google Drive so your cached embeddings
and trained models survive a disconnect.

> For notebook 04 (LoRA), also enable a GPU: **Runtime → Change runtime type → T4 GPU**.

In [ ]:
# Clone the repo so kv_common.py and artifacts live in one place.
import os
if not os.path.exists("KVCacheCompression"):
    !git clone -q https://github.com/yoshikodes/KVCacheCompression.git
# Work inside the notebooks folder (edit if your notebooks live elsewhere).
if os.path.basename(os.getcwd()) != "notebooks":
    %cd KVCacheCompression/notebooks
print("cwd:", os.getcwd())
assert os.path.exists("kv_common.py"), "kv_common.py not found"


In [ ]:
# Install packages not preinstalled on Colab.
# If you get an import error right after this, do Runtime -> Restart, then re-run from the top.
!pip install -q sentence-transformers lightgbm scikit-learn scipy requests joblib

In [ ]:
# Optional but recommended: mount Drive so artifacts/ persists across sessions.
# Without this, trained models and cached embeddings are lost on disconnect,
# and notebook 05 will only see models trained in the current session.
USE_DRIVE = True  # set False to keep everything ephemeral

import os
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        ART_DIR = "/content/drive/MyDrive/kv_artifacts"
        os.makedirs(ART_DIR, exist_ok=True)
        # Link ./artifacts -> Drive so all the notebook's save paths persist.
        if os.path.islink("artifacts") or os.path.exists("artifacts"):
            if not os.path.islink("artifacts"):
                import shutil; shutil.rmtree("artifacts", ignore_errors=True)
        if not os.path.exists("artifacts"):
            os.symlink(ART_DIR, "artifacts")
        print("artifacts -> ", os.path.realpath("artifacts"))
    except Exception as e:
        print("Drive mount skipped:", e)
        os.makedirs("artifacts", exist_ok=True)
else:
    os.makedirs("artifacts", exist_ok=True)

In [ ]:
import numpy as np, pandas as pd, os, joblib
import kv_common as kv

os.makedirs("artifacts", exist_ok=True)
df = kv.load_data()
train_df, test_df = kv.stratified_split(df)

LOG_SPACE = True
y_train = kv.get_targets(train_df, log_space=LOG_SPACE)
y_test  = kv.get_targets(test_df,  log_space=LOG_SPACE)
baseline = kv.baseline_predict_mean(y_train, len(y_test))
baseline_metrics = kv.evaluate(y_test, baseline)
print("Mean-baseline OVERALL MAE_log:",
      round(baseline_metrics[baseline_metrics.setting=='OVERALL'].MAE_log.iloc[0], 4))


## Encode prompts with frozen ModernBERT
Embeddings are cached to `artifacts/` and reused by notebook 03.

In [ ]:
from sentence_transformers import SentenceTransformer
import torch

ENCODER_NAME = "nomic-ai/modernbert-embed-base"  # ModernBERT-based, 8192 ctx
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| encoder:", ENCODER_NAME)

encoder = SentenceTransformer(ENCODER_NAME, device=device)

# Cache embeddings to disk so you only encode once across notebooks.
CACHE = "artifacts/embeddings_modernbert.npz"
if os.path.exists(CACHE):
    z = np.load(CACHE, allow_pickle=True)
    emb_train, emb_test = z["train"], z["test"]
    print("loaded cached embeddings", emb_train.shape, emb_test.shape)
else:
    emb_train = encoder.encode(train_df["prompt"].tolist(),
                               batch_size=32, show_progress_bar=True,
                               convert_to_numpy=True)
    emb_test  = encoder.encode(test_df["prompt"].tolist(),
                               batch_size=32, show_progress_bar=True,
                               convert_to_numpy=True)
    np.savez(CACHE, train=emb_train, test=emb_test)
    print("encoded + cached", emb_train.shape, emb_test.shape)

## Head A — Ridge regression
Multi-output by default; a strong linear baseline on embeddings.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

ridge = make_pipeline(StandardScaler(), Ridge(alpha=10.0))
ridge.fit(emb_train, y_train)
pred_ridge = ridge.predict(emb_test)
metrics_ridge = kv.evaluate(y_test, pred_ridge)
print(metrics_ridge.round(4).to_string(index=False))
kv.print_comparison(metrics_ridge, baseline_metrics)

## Head B — LightGBM on embeddings

In [ ]:
import lightgbm as lgb
from sklearn.multioutput import MultiOutputRegressor

gbm = MultiOutputRegressor(lgb.LGBMRegressor(
    n_estimators=400, num_leaves=31, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.8,
    random_state=kv.RANDOM_STATE, verbose=-1))
gbm.fit(emb_train, y_train)
pred_gbm = gbm.predict(emb_test)
metrics_gbm = kv.evaluate(y_test, pred_gbm)
print(metrics_gbm.round(4).to_string(index=False))
kv.print_comparison(metrics_gbm, baseline_metrics)

## Save the better head

In [ ]:
r2_ridge = metrics_ridge[metrics_ridge.setting=="OVERALL"].R2_log.iloc[0]
r2_gbm   = metrics_gbm[metrics_gbm.setting=="OVERALL"].R2_log.iloc[0]
best, best_name, best_metrics = ((ridge,"ridge",metrics_ridge) if r2_ridge>=r2_gbm
                                 else (gbm,"gbm",metrics_gbm))
joblib.dump({"model": best, "head": best_name, "encoder": ENCODER_NAME,
             "settings": kv.SETTINGS, "log_space": LOG_SPACE},
            "artifacts/model2_frozen_emb.joblib")
best_metrics.to_csv("artifacts/model2_frozen_emb_metrics.csv", index=False)
print("saved best head:", best_name)